# Qwen2.5-VL-7B-Instruct Abliterated (Multimodal Vision + Code + Text)
This server runs **`huihui-ai/Qwen2.5-VL-7B-Instruct-abliterated`** on Google Colab's GPU in 4-bit (using ~6 GB VRAM).
- **Vision:** Analyzes diagrams, screenshots, UI mockups, documents, handwriting, and charts.
- **Coding:** State-of-the-art coding and converts UI designs into clean code.
- **Reasoning:** Superior multimodal logic and math solving.
- **Unrestricted:** Refusal directions have been abliterated.
- **Google Drive:** Downloaded weights persist in your Google Drive.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
drive_cache_dir = '/content/drive/MyDrive/colab_model_cache'
os.makedirs(drive_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = drive_cache_dir
print(f"Google Drive Model Storage: {drive_cache_dir}")

In [ ]:
# Step 2: Install required packages
!pip install -q transformers accelerate bitsandbytes torchvision qwen-vl-utils fastapi uvicorn pycloudflared pydantic pillow

In [ ]:
# Step 3: Load Qwen2.5-VL-7B-Instruct-abliterated in 4-bit into Colab GPU
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "huihui-ai/Qwen2.5-VL-7B-Instruct-abliterated"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Loading {MODEL_ID} into GPU with 4-bit quantization...")
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=drive_cache_dir)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=drive_cache_dir
)
print("Model loaded successfully into Colab GPU!")

In [ ]:
# Step 4: Run Multimodal API Server with Cloudflare Tunnel
import io
import base64
import threading
from typing import Optional
from PIL import Image
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare
from qwen_vl_utils import process_vision_info
from transformers import TextIteratorStreamer

app = FastAPI()

class Query(BaseModel):
    prompt: str
    image_base64: Optional[str] = None
    image_url: Optional[str] = None
    max_tokens: int = 1024

def prepare_messages(q: Query):
    content = []
    if q.image_base64:
        img_data = base64.b64decode(q.image_base64)
        image = Image.open(io.BytesIO(img_data)).convert("RGB")
        content.append({"type": "image", "image": image})
    elif q.image_url:
        content.append({"type": "image", "image": q.image_url})
    
    content.append({"type": "text", "text": q.prompt})
    return [{"role": "user", "content": content}]

@app.post("/generate")
def generate(q: Query):
    messages = prepare_messages(q)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=q.max_tokens)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
    return {"response": output_text[0]}

@app.post("/stream")
def stream(q: Query):
    messages = prepare_messages(q)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(inputs, streamer=streamer, max_new_tokens=q.max_tokens)

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    def event_stream():
        for new_text in streamer:
            yield new_text

    return StreamingResponse(event_stream(), media_type="text/plain")

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_server, daemon=True).start()

# Expose via Cloudflare Tunnel
public_url = try_cloudflare(port=8000)
print("\n=======================================================")
print(f"MULTIMODAL COLAB SERVER READY: {public_url.tunnel}")
print("=======================================================")
print("Run in your local CLI:")
print(f"colab --set-url {public_url.tunnel}")
print("colab --image path/to/image.png \"Explain this and write code\"")
print("colab --chat")